# Churn exploration
Quick look at the churn data before modelling.

In [1]:
%matplotlib inline
import pandas as pd
import numpy as np
from pathlib import Path

## Load
Raw exports from the warehouse.

In [2]:
DATA_DIR = Path('data')
SEED = 42

In [3]:
events = pd.read_csv(DATA_DIR / 'events.csv')
events.shape

(41203, 6)

In [4]:
events.head()

customer_id,amount
1,9.5
2,3.0


## Features
Per-customer aggregates.

In [6]:
def build_features(df, seed):
    g = df.groupby('customer_id')
    feats = g.agg(n=('amount', 'size'), total=('amount', 'sum'))
    feats['avg'] = feats['total'] / feats['n']
    return feats.sample(frac=1, random_state=seed)

feats = build_features(events, SEED)
print(f'{len(feats)} customers')

3120 customers


In [7]:
feats['label'] = labels['churned']   # labels is defined further down
feats['label'].mean()

0.1831

In [5]:
labels = pd.read_csv(DATA_DIR / 'labels.csv').set_index('customer_id')
labels.churned.value_counts()

0    2549
1     571
Name: churned, dtype: int64

## Plots

In [8]:
ax = feats.plot.scatter(x='n', y='total')
ax.set_title('spend vs activity');

In [9]:
%%time
model_input = feats.dropna()
model_input.describe()?

KeyError: 'avg'

In [ ]:
!pip install lightgbm -q

In [10]:
%%bash
echo done

done


## Save

In [11]:
model_input.to_parquet(DATA_DIR / 'model_input.parquet')
print('wrote', len(model_input))

wrote 3120
